# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jazaalee/FlyRank-StarterNotebook/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

Lane: Refresh / Content Opportunity Scoring

I picked this lane because it is basically what I was already doing in my earlier notebook work, with a proper name on it now. I had built a simple scoring rule to flag pages that seemed stale but were still getting views, trained a small decision tree to try to do the same thing on its own, and then tested it properly by making sure the same client's pages weren't showing up in both my training data and my testing data (so the model wasn't just memorizing).

The reason this lane makes sense for me is that it matches a real problem: a content team doesn't have time to go through every single page, so what actually helps them is a short list of "review these first" pages, with a reason attached to each one.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*
**What decision does this improve?**
It helps decide which pages a content team should look at first when they're trying to figure out what to refresh, rewrite, or leave alone. Right now that decision is basically a guess. Someone skims through pages and picks a few that "feel" outdated. This gives them an actual ranked list instead.

**Who acts on it?**
A content reviewer or strategist — someone who has limited time and need to perform within that time frame. They'd use the list to decide what to update first, not to make the update themselves.

**What does a wrong recommendation cost?**
If the list flags a page as "review this" and it turns out fine, that's not a big issue — the reviewer just spends a bit of time confirming it's okay and moves on. The bigger risk is the opposite: if a page that's actually declining gets ranked low and never gets reviewed, it keeps losing traffic quietly while nobody notices. So a false "don't worry about this one" is more expensive than a false "maybe check this."

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*
**Quick look at the data**

I loaded the starter dataset (30,000 rows, 44 columns) and checked a few numbers to see if this lane is actually worth pursuing.

Only 17 pages (0.1%) are both stale (not updated in 6+ months) and still getting real traffic (500+ impressions), so the "obvious" stale-and-visible case is rare. But 54.2% of all pages are currently flagged as declining, which is a much bigger pool than I expected. That gap tells me most of the decline isn't just simple staleness. there's more going on that a smarter method could help sort through, rather than a plain rule catching everything.

My simple hand-written rule already gets a Precision@50 of 0.680 — meaning about 34 out of the top 50 pages it flags are genuinely declining. That's a decent starting point, but earlier testing (from my Week 1-2 notebooks) showed a decision tree can beat this same kind of rule, which means the improvement we have gained is going from "good guess" to "defensible ranked list."

In [4]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/jazaalee/FlyRank-StarterNotebook"
REPO_DIR = "FlyRank-StarterNotebook"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print("Now in:", os.getcwd())

import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape[0], "rows,", df.shape[1], "columns")

# Build the label and a few reference columns, same as earlier work
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Number 1: how many pages are stale AND still visible — the core opportunity this lane targets
stale_visible = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).sum()
pct_stale_visible = round(100 * stale_visible / len(df), 1)
print(f"Stale-but-visible pages: {stale_visible} ({pct_stale_visible}% of the dataset)")

# Number 2: how many pages are actively declining
declining_pct = round(100 * df["is_declining_label"].mean(), 1)
print(f"Pages currently declining: {declining_pct}%")

# Number 3: baseline rule vs a simple tree, using Precision@50 (from earlier work)
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

y = df["is_declining_label"].values
hand_rule_p50 = precision_at_k(df["hand_rule_score"], y, 50)
print(f"Hand-rule Precision@50: {hand_rule_p50:.3f}")


Now in: /content/FlyRank-StarterNotebook
30000 rows, 44 columns
Stale-but-visible pages: 17 (0.1% of the dataset)
Pages currently declining: 54.2%
Hand-rule Precision@50: 0.680


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**Careful words: what I can and can't claim**

**What this work can say:** which pages, based on the patterns in this data, look like good candidates to review first. I can say a page is "stale and still visible" or "declining while demand exists" because those are things *I can directly measure.* I can also say my ranking is better or worse than a simple rule, because I can test that with Precision@K. All of this is observed and directional. It is evidence pointing toward where attention is probably worth spending, not a certainty.

**What this work can't say:** that refreshing a page will actually fix it. I have no way to prove that with this data. That would need an actual experiment, like updating some pages and comparing them to ones left alone. I also can't say anything about how Google's ranking algorithm actually works, or claim I've "predicted" what Google will do. My model only ever sees search and engagement numbers that already happened, it's not looking inside Google's system, so it can't explain why something ranks the way it does, only flag pages that look worth a human's attention.

This is decision support, not a guarantee. It narrows down where to look, whereas the actual judgment still belongs to a person

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.